# 🧠 Deep Reinforcement Learning — Doom Agent (SS2025)

Welcome to the last assignment for the **Deep Reinforcement Learning** course (SS2025). In this notebook, you'll implement and train a reinforcement learning agent to play **Doom**.

You will:
- Set up a custom VizDoom environment with shaped rewards
- Train an agent using an approach of your choice
- Track reward components across episodes
- Evaluate the best model
- Visualize performance with replays and GIFs
- Export the trained agent to ONNX to submit to the evaluation server

In [ ]:
# Run this once on Google Colab (clones the repo + installs dependencies).
# Locally, use the provided environment.yaml instead.
import os, platform

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !apt-get update -qq
    !apt-get install -y -qq swig cmake zlib1g-dev libjpeg-dev xvfb ffmpeg xorg-dev python3-opengl libboost-all-dev libsdl2-dev
    !pip install -q torch numpy matplotlib vizdoom==1.3.0 portpicker gymnasium==1.2.3 onnx onnx2pytorch einops stable-baselines3==2.8.0 tensorboard
    !git clone https://$token@github.com/gerkone/jku.wad.git
    %cd jku.wad

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print("Colab:", IN_COLAB, "| OS:", platform.system())

In [ ]:
from typing import Dict, Tuple

import os
import glob
import json
from copy import deepcopy
from datetime import datetime

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions.categorical import Categorical
from matplotlib import pyplot as plt

from doom_arena import VizdoomMPEnv
from doom_arena.reward import VizDoomReward
from doom_arena.render import render_episode
from IPython.display import HTML

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float32
print("Device:", device)

## Environment configuration

ViZDoom supports multiple visual buffers that can be used as input for training agents. Each buffer provides different information about the game environment, as seen from left to right:


Screen
- The default first-person RGB view seen by the agent.

Labels
- A semantic map where each pixel is tagged with an object ID (e.g., enemy, item, wall).

Depth
- A grayscale map showing the distance from the agent to surfaces in the scene.

Automap
- A top-down schematic view of the map, useful for global navigation tasks.

![buffers gif](https://vizdoom.farama.org/_images/vizdoom-demo.gif)

In [ ]:
USE_GRAYSCALE = False  # RGB screen (3 channels)

PLAYER_CONFIG = {
    "algo_type": "POLICY",               # the eval server samples actions from the policy logits
    "n_stack_frames": 1,
    "extra_state": ["labels", "depth"],  # observation = RGB(3) + labels(1) + depth(1)
    "hud": "none",
    "crosshair": True,
    "screen_format": 8 if USE_GRAYSCALE else 0,
}

In [ ]:
# Environment
N_STACK_FRAMES  = 1
NUM_BOTS        = 4        # opponents during training and evaluation
EPISODE_TIMEOUT = 2000     # frames per episode

# Behaviour cloning (warm-start)
BC_DATA_DIR     = "bc_data_collection"   # recorded demonstrations (ep_*.npz)
BC_CACHE_DIR    = "bc_cache"             # resized uint8 frame cache (built once)
BC_EPOCHS       = 8
BC_BATCH_SIZE   = 256
BC_LR           = 1e-3
BC_VAL_FRACTION = 0.05
BC_NUM_WORKERS  = 6

# PPO
EPISODES      = 3000      # PPO trains EPISODES * EPISODE_TIMEOUT environment steps
LEARNING_RATE = 1e-4
BATCH_SIZE    = 256
N_EPOCHS      = 4
GAMMA         = 0.99
N_STEPS       = 2048
GAE_LAMBDA    = 0.95
CLIP_RANGE    = 0.2
CLIP_RANGE_VF = 0.2       # value-function clipping
ENT_COEF      = 0.0
VF_COEF       = 0.5
MAX_GRAD_NORM = 0.5
TARGET_KL     = 0.03      # keep PPO updates close to the cloned policy

# Checkpointing / evaluation
WARMSTART_FROM_BC = True
CKPT_FREQ         = 20_000
EVAL_FREQ         = 20_000
N_EVAL_CB         = 5
N_EVAL_EPISODES   = 10

RUN_DIR  = "runs"
CKPT_DIR = "checkpoints"

## Reward function
In this task, you will define a shaped reward function to guide the agent's learning. The function is called at every step and receives the current and previous game variables (e.g., number of frags, hits taken, health).

Your goal is to combine these into a meaningful reward signal that encourages desirable behavior, such as:

- Rewarding frags (enemy kills)

- Rewarding accuracy (hitting enemies)

- Penalizing damage taken

- (Optional) Encouraging survival, ammo efficiency, etc.

You can return multiple reward components, which are automatically summed during training. Consider the class below as a great starting point!

In [ ]:
class YourReward(VizDoomReward):
    def __init__(self, num_players: int):
        super().__init__(num_players)

    def __call__(
        self,
        vizdoom_reward: float,
        game_var: Dict[str, float],
        game_var_old: Dict[str, float],
        player_id: int,
    ) -> Tuple[float, float, float]:
        """
        Shaped reward used by both training and evaluation.
        *  +100  for every new frag
        *  +2    for every hit landed
        *  -0.1  for every hit taken
        """
        self._step += 1
        _ = vizdoom_reward, player_id  # unused

        rwd_hit       =  2.0  * (game_var["HITCOUNT"]   - game_var_old["HITCOUNT"])
        rwd_hit_taken = -0.1  * (game_var["HITS_TAKEN"] - game_var_old["HITS_TAKEN"])
        rwd_frag      = 100.0 * (game_var["FRAGCOUNT"]  - game_var_old["FRAGCOUNT"])

        return rwd_hit, rwd_hit_taken, rwd_frag

In [ ]:
reward_fn = YourReward(num_players=1)

# Reference environment (raw VizdoomMPEnv), used for the ONNX dummy input and the replay
# videos. The SB3 training environment is built separately via make_doom_env below.
env = VizdoomMPEnv(
    num_players=1,
    num_bots=NUM_BOTS,
    bot_skill=0,
    doom_map="ROOM",  # simple, small map; other options: TRNM, TRNMBIG
    extra_state=PLAYER_CONFIG["extra_state"],
    episode_timeout=EPISODE_TIMEOUT,
    n_stack_frames=PLAYER_CONFIG["n_stack_frames"],
    crosshair=PLAYER_CONFIG["crosshair"],
    hud=PLAYER_CONFIG["hud"],
    screen_format=PLAYER_CONFIG["screen_format"],
    reward_fn=reward_fn,
)
print("reference env | obs:", env.observation_space.shape, "| actions:", env.action_space.n)

## Agent

Implement **your own agent** in the code cell that follows.

* In `agents/dqn.py` and `agents/ppo.py` you’ll find very small **skeletons**—they compile but are meant only as reference or quick tests.
  Feel free to open them, borrow ideas, extend them, or ignore them entirely.
* The notebook does **not** import those files automatically; whatever class you define in the next cell is the one that will be trained.
* You may keep the DQN interface, switch to PPO, or try something else.
* Tweak any hyper-parameters (`PLAYER_CONFIG`, optimiser, etc.) and document what you tried.

In [ ]:
# We use behaviour cloning to warm-start a shared Actor-Critic network, then fine-tune
# it with PPO (Stable-Baselines3). Observations are float32 in [0, 1] at 128x128.
# The feature extractor resizes its input to 128x128 (nearest) as its first operation,
# so the network is robust to the input resolution.
import gymnasium as gym
from gymnasium.spaces import Box, Discrete
from stable_baselines3 import PPO
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3.common.monitor import Monitor

OBS_SHAPE = (5, 128, 128)  # env output: RGB(3) + labels(1) + depth(1)
PROC_SIZE = (128, 128)     # network input after the internal resize (== server)
N_ACTIONS = 8              # Discrete(8): NOOP + 7 buttons


class LayerNorm2d(nn.Module):
    """LayerNorm over the channel dimension (ConvNeXt-style), resolution-agnostic."""

    def __init__(self, c):
        super().__init__()
        self.ln = nn.LayerNorm(c)

    def forward(self, x):
        x = x.permute(0, 2, 3, 1)
        x = self.ln(x)
        return x.permute(0, 3, 1, 2)


class DoomCNN(BaseFeaturesExtractor):
    """Resize to PROC_SIZE (nearest) -> 3 x [Conv -> LayerNorm -> SiLU] -> Flatten.
    features_dim is inferred from a dummy forward pass."""

    def __init__(self, observation_space, channels=(16, 32, 32), proc_size=PROC_SIZE):
        c_in = observation_space.shape[0]
        cnn = nn.Sequential(
            nn.Conv2d(c_in, channels[0], 8, stride=4), LayerNorm2d(channels[0]), nn.SiLU(),
            nn.Conv2d(channels[0], channels[1], 4, stride=2), LayerNorm2d(channels[1]), nn.SiLU(),
            nn.Conv2d(channels[1], channels[2], 3, stride=1), LayerNorm2d(channels[2]), nn.SiLU(),
            nn.Flatten(),
        )
        with torch.no_grad():
            dummy = F.interpolate(torch.zeros(1, *observation_space.shape), size=proc_size)
            n_flat = cnn(dummy).shape[1]
        super().__init__(observation_space, n_flat)
        self.proc_size = proc_size
        self.cnn = cnn

    def forward(self, x):
        return self.cnn(F.interpolate(x, size=self.proc_size))  # nearest interpolation


class ExportNet(nn.Module):
    """obs (B, C, H, W) in [0, 1] -> action logits (B, 8). Reuses the trained policy
    modules; no /255 because the eval server already feeds observations in [0, 1]."""

    def __init__(self, policy):
        super().__init__()
        self.features_extractor = policy.features_extractor
        self.mlp_extractor = policy.mlp_extractor
        self.action_net = policy.action_net

    def forward(self, x):
        f = self.features_extractor(x)
        return self.action_net(self.mlp_extractor.forward_actor(f))


class DoomSB3Wrapper(gym.Env):
    """Single-player VizdoomMPEnv -> Gymnasium env for SB3: unwraps the list-wrapped
    obs/reward, converts to float32 numpy, and maps the 4-tuple step to the 5-tuple API
    (timeout -> truncated, so PPO bootstraps the value at the cutoff)."""

    metadata = {"render_modes": []}

    def __init__(self, mp_env):
        super().__init__()
        self.mp_env = mp_env
        self.observation_space = Box(0.0, 1.0, OBS_SHAPE, np.float32)
        self.action_space = Discrete(mp_env.action_space.n)

    @staticmethod
    def _obs(obs_list):
        o = obs_list[0]
        if hasattr(o, "detach"):
            o = o.detach().cpu().numpy()
        return np.ascontiguousarray(o, dtype=np.float32)

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        return self._obs(self.mp_env.reset()), {}

    def step(self, action):
        obs, rwds, done, info = self.mp_env.step(int(action))
        info = dict(info) if info else {}
        try:  # expose frags for logging
            info["frags"] = float(self.mp_env.envs[0].unwrapped._game_vars.get("FRAGCOUNT", 0.0))
        except Exception:
            pass
        return self._obs(obs), float(rwds[0]), False, bool(done), info

    def close(self):
        try:
            self.mp_env.close()
        except Exception:
            pass


def make_doom_env(num_bots=NUM_BOTS, seed=1337, episode_timeout=EPISODE_TIMEOUT, monitor=True):
    """SB3-ready single-player ROOM env with the 5-channel observation."""
    mp = VizdoomMPEnv(
        num_players=1, num_bots=num_bots, bot_skill=0, doom_map="ROOM",
        extra_state=PLAYER_CONFIG["extra_state"], episode_timeout=episode_timeout,
        n_stack_frames=PLAYER_CONFIG["n_stack_frames"], crosshair=PLAYER_CONFIG["crosshair"],
        hud=PLAYER_CONFIG["hud"], screen_format=PLAYER_CONFIG["screen_format"], seed=seed,
    )
    e = DoomSB3Wrapper(mp)
    return Monitor(e, info_keywords=("frags",)) if monitor else e

In [ ]:
os.makedirs(RUN_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

train_env = make_doom_env(num_bots=NUM_BOTS, seed=1234, episode_timeout=EPISODE_TIMEOUT)

policy_kwargs = dict(
    features_extractor_class=DoomCNN,
    net_arch=dict(pi=[32], vf=[16]),   # actor: feat->32->8 ; critic: feat->16->1
    activation_fn=nn.ReLU,
    normalize_images=False,            # observations are already float [0, 1]
    optimizer_kwargs=dict(eps=1e-5),
)

model = PPO(
    "MlpPolicy",
    train_env,
    learning_rate=LEARNING_RATE,
    n_steps=N_STEPS,
    batch_size=BATCH_SIZE,
    n_epochs=N_EPOCHS,
    gamma=GAMMA,
    gae_lambda=GAE_LAMBDA,
    clip_range=CLIP_RANGE,
    clip_range_vf=CLIP_RANGE_VF,
    ent_coef=ENT_COEF,
    vf_coef=VF_COEF,
    max_grad_norm=MAX_GRAD_NORM,
    target_kl=TARGET_KL,
    policy_kwargs=policy_kwargs,
    tensorboard_log=RUN_DIR,
    device=device,
    verbose=1,
)
print(f"PPO ready | features_dim={model.policy.features_extractor.features_dim} "
      f"| params={sum(p.numel() for p in model.policy.parameters())/1e6:.3f} M")

In [ ]:
# Utilities: policy save/load, best-so-far tracking, ONNX export, evaluation, replay.
import onnx

BC_POLICY   = os.path.join(CKPT_DIR, "bc_policy.pt")     # BC warm-start weights
BEST_POLICY = os.path.join(CKPT_DIR, "best_policy.pt")   # best policy so far (BC or PPO)
BEST_META   = os.path.join(CKPT_DIR, "best_meta.json")


def save_policy(policy, path):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    torch.save(policy.state_dict(), path)


def load_policy(policy, path):
    policy.load_state_dict(torch.load(path, map_location=device))
    return policy


def get_best_score():
    if os.path.exists(BEST_META):
        try:
            return float(json.load(open(BEST_META)).get("score", float("-inf")))
        except Exception:
            return float("-inf")
    return float("-inf")


def update_best(policy, score, tag):
    """Persist `policy` as the new best only if `score` beats the stored best."""
    prev = get_best_score()
    if score > prev:
        save_policy(policy, BEST_POLICY)
        json.dump({"score": float(score), "tag": tag, "time": datetime.now().isoformat()},
                  open(BEST_META, "w"), indent=2)
        print(f"  new best ({tag}): {score:.2f} (prev {prev:.2f})")
        return True
    print(f"  kept best {prev:.2f} (this {tag} = {score:.2f})")
    return False


def load_best(policy):
    if os.path.exists(BEST_POLICY):
        load_policy(policy, BEST_POLICY)
        print(f"loaded best policy (score {get_best_score():.2f})")
    return policy


def _strip_identity(m):
    # torch can de-duplicate equal-valued parameters into Identity alias nodes, which the
    # server's onnx2pytorch loader cannot handle on Conv inputs; inline them away.
    g = m.graph
    out_names = {o.name for o in g.output}
    alias = {n.output[0]: n.input[0] for n in g.node
             if n.op_type == "Identity" and n.output[0] not in out_names}

    def resolve(name):
        seen = set()
        while name in alias and name not in seen:
            seen.add(name)
            name = alias[name]
        return name

    for n in g.node:
        if n.op_type == "Identity" and n.output[0] in alias:
            continue
        for i, inp in enumerate(n.input):
            n.input[i] = resolve(inp)
    keep = [n for n in g.node if not (n.op_type == "Identity" and n.output[0] in alias)]
    del g.node[:]
    g.node.extend(keep)
    return m


def onnx_dump(env, model, config, filename="models/model.onnx"):
    os.makedirs(os.path.dirname(filename) or ".", exist_ok=True)
    init_state = env.reset()[0].unsqueeze(0)   # (1, 5, 128, 128) float [0, 1]
    net = deepcopy(model).cpu().eval()
    torch.onnx.export(
        net, args=init_state.cpu(), f=filename, export_params=True,
        opset_version=11, do_constant_folding=False,
        input_names=["input"], output_names=["output"],
        dynamic_axes={"input": {0: "batch", 2: "h", 3: "w"}, "output": {0: "batch"}},
        dynamo=False,
    )
    m = _strip_identity(onnx.load(filename))
    meta = m.metadata_props.add()
    meta.key = "config"
    meta.value = json.dumps(config)
    onnx.save(m, filename)
    print(f"exported -> {filename}")


@torch.no_grad()
def evaluate(model, n_episodes=10, seed=None):
    """Server-like evaluation: sample actions from the policy logits over `n_episodes`
    fresh 4-bot ROOM episodes. Returns the mean reward."""
    model.eval()
    rng = np.random.default_rng(seed)
    rewards = []
    for ep in range(n_episodes):
        ep_seed = int(rng.integers(0, 10_000_000))
        eval_env = make_doom_env(num_bots=NUM_BOTS, seed=ep_seed,
                                 episode_timeout=EPISODE_TIMEOUT, monitor=False)
        obs, _ = eval_env.reset()
        done, total = False, 0.0
        while not done:
            obs_t = torch.as_tensor(obs, device=device, dtype=DTYPE).unsqueeze(0)
            action = int(Categorical(logits=model(obs_t)).sample()[0])
            obs, r, term, trunc, _ = eval_env.step(action)
            total += r
            done = term or trunc
        eval_env.close()
        rewards.append(total)
        print(f"  episode {ep+1:02}/{n_episodes}: seed={ep_seed} reward={total:.1f}")
    mean = float(np.mean(rewards))
    print(f"mean reward over {n_episodes} episodes: {mean:.2f}")
    return mean


def record_and_render(model, env, subsample=6):
    """Play one episode (sampling, like the server) and return an inline animation."""
    model.eval()
    env.enable_replay()
    obs = env.reset()[0]
    done = False
    while not done:
        obs_t = torch.as_tensor(obs, device=device, dtype=DTYPE).unsqueeze(0)
        with torch.no_grad():
            a = int(Categorical(logits=model(obs_t)).sample()[0])
        step = env.step(a)
        obs, done = step[0][0], step[2]
    env.disable_replay()
    return HTML(render_episode(env.get_player_replays(), subsample=subsample).to_jshtml())

## Behaviour cloning warm-start

Before reinforcement learning, we pre-train the policy on recorded demonstrations
(`bc_data_collection/ep_*.npz`) using supervised cross-entropy on the demonstrated
actions. This gives PPO a competent starting point instead of a random policy.

Record your own demonstrations by running `play_doom.py` in a terminal (not in the
notebook); each run is saved as `bc_data_collection/ep_XXXX.npz`.

In [ ]:
# Build (once) a uint8 cache of all demonstration frames, resized to the model input
# size (128x128, nearest) so it stays compact. Channels [0:5] = screen(3)+labels(1)+depth(1).
from pathlib import Path
from torch.utils.data import Dataset, DataLoader, random_split

BC_SAVED_BUFFERS = ["screen", "labels", "depth", "automap"]  # order stored in the .npz
BC_TRAIN_BUFFERS = ["screen", *PLAYER_CONFIG.get("extra_state", [])]


def bc_channel_indices(saved_buffers, train_buffers):
    widths = {"screen": 1 if USE_GRAYSCALE else 3, "labels": 1, "depth": 1, "automap": 3}
    saved, offset = {}, 0
    for name in saved_buffers:
        saved[name] = list(range(offset, offset + widths[name]))
        offset += widths[name]
    idx = []
    for name in train_buffers:
        idx.extend(saved[name])
    return idx


def build_bc_memmap(data_dir, cache_dir, saved_buffers, train_buffers,
                    proc_size=PROC_SIZE, frame_stride=1, chunk=512):
    out = Path(cache_dir); out.mkdir(exist_ok=True)
    H, W = proc_size
    states_path = out / f"states_u8_{H}x{W}.npy"
    actions_path = out / f"actions_{H}x{W}.npy"
    if states_path.exists() and actions_path.exists():
        print(f"[cache exists] {states_path}")
        return states_path, actions_path

    ch_idx = bc_channel_indices(saved_buffers, train_buffers)
    files = sorted(glob.glob(os.path.join(data_dir, "ep_*.npz")))
    counts, good = [], []
    for f in files:
        try:
            counts.append(len(np.load(f)["actions"][::frame_stride]))
            good.append(f)
        except Exception as e:
            print(f"  skip {os.path.basename(f)}: {str(e)[:50]}")
    N, C = sum(counts), len(ch_idx)
    print(f"building {H}x{W} cache: N={N} C={C} from {len(good)} files")
    states = np.lib.format.open_memmap(states_path, mode="w+", dtype=np.uint8, shape=(N, C, H, W))
    actions = np.empty(N, dtype=np.int64)
    off = 0
    for f, T in zip(good, counts):
        d = np.load(f)
        arr = d["states"][::frame_stride]
        for i in range(0, T, chunk):
            ck = torch.from_numpy(arr[i:i + chunk][:, ch_idx]).float()
            ck = F.interpolate(ck, size=proc_size)
            states[off + i:off + i + ck.shape[0]] = (ck.clamp_(0, 1) * 255).to(torch.uint8).numpy()
        actions[off:off + T] = d["actions"][::frame_stride].astype(np.int64)
        off += T
        del d, arr
    states.flush()
    np.save(actions_path, actions)
    print(f"  wrote {states_path} ({states.nbytes / 1e9:.1f} GB)")
    return states_path, actions_path


class BCMemmapDataset(Dataset):
    """Returns (uint8 state (C, H, W), action); converted to float/255 in the training loop."""

    def __init__(self, states_path, actions_path):
        self.states = np.load(states_path, mmap_mode="r")
        self.actions = np.load(actions_path)

    def __len__(self):
        return len(self.actions)

    def __getitem__(self, i):
        return torch.from_numpy(np.array(self.states[i])), int(self.actions[i])


_sp, _ap = build_bc_memmap(BC_DATA_DIR, BC_CACHE_DIR, BC_SAVED_BUFFERS, BC_TRAIN_BUFFERS)
bc_dataset = BCMemmapDataset(_sp, _ap)
bc_val_size = max(1, int(len(bc_dataset) * BC_VAL_FRACTION))
bc_train_size = len(bc_dataset) - bc_val_size
bc_train_dataset, bc_val_dataset = random_split(
    bc_dataset, [bc_train_size, bc_val_size], generator=torch.Generator().manual_seed(42))
bc_train_loader = DataLoader(bc_train_dataset, batch_size=BC_BATCH_SIZE, shuffle=True,
                             num_workers=BC_NUM_WORKERS, pin_memory=True, drop_last=True)
bc_val_loader = DataLoader(bc_val_dataset, batch_size=BC_BATCH_SIZE, shuffle=False,
                           num_workers=2, pin_memory=True)
print(f"BC dataset: {len(bc_dataset)} frames | train={bc_train_size} val={bc_val_size} | "
      f"action histogram={np.bincount(bc_dataset.actions, minlength=N_ACTIONS).tolist()}")

In [ ]:
# Behaviour cloning: supervised cross-entropy on the shared policy network.
from torch.utils.tensorboard import SummaryWriter


@torch.no_grad()
def bc_evaluate(policy, loader):
    policy.set_training_mode(False)
    n = correct = 0
    loss_sum = 0.0
    for s_u8, a in loader:
        s = s_u8.to(device).float().div_(255.0)
        a = a.to(device)
        logits = policy.get_distribution(s).distribution.logits
        loss_sum += F.cross_entropy(logits, a, reduction="sum").item()
        correct += (logits.argmax(-1) == a).sum().item()
        n += a.numel()
    return loss_sum / n, correct / n


def bc_train(policy, train_loader, val_loader, epochs, lr):
    writer = SummaryWriter(os.path.join(RUN_DIR, f"bc_{datetime.now():%Y%m%d_%H%M%S}"), flush_secs=10)
    opt = torch.optim.Adam(policy.parameters(), lr=lr)
    step = 0
    for ep in range(epochs):
        policy.set_training_mode(True)
        run_loss, seen = 0.0, 0
        for s_u8, a in train_loader:
            s = s_u8.to(device, non_blocking=True).float().div_(255.0)
            a = a.to(device, non_blocking=True)
            loss = -policy.get_distribution(s).log_prob(a).mean()
            opt.zero_grad()
            loss.backward()
            opt.step()
            run_loss += loss.item() * a.numel()
            seen += a.numel()
            step += 1
            if step % 20 == 0:
                writer.add_scalar("bc/train_loss", loss.item(), step)
                writer.flush()
        vloss, vacc = bc_evaluate(policy, val_loader)
        writer.add_scalar("bc/val_loss", vloss, step)
        writer.add_scalar("bc/val_acc", vacc, step)
        writer.flush()
        print(f"  BC epoch {ep+1}/{epochs}: train_loss={run_loss/seen:.4f} "
              f"val_loss={vloss:.4f} val_acc={vacc:.3f}")
    writer.close()


_bl, _ba = bc_evaluate(model.policy, bc_val_loader)
print(f"baseline (untrained): val_loss={_bl:.4f} val_acc={_ba:.3f}")
bc_train(model.policy, bc_train_loader, bc_val_loader, epochs=BC_EPOCHS, lr=BC_LR)
save_policy(model.policy, BC_POLICY)
print("saved BC policy ->", BC_POLICY)

# Free the dataset / DataLoader workers before PPO.
import gc
for _n in ["bc_train_loader", "bc_val_loader", "bc_train_dataset", "bc_val_dataset", "bc_dataset"]:
    globals().pop(_n, None)
gc.collect()

In [ ]:
# Evaluate the behaviour-cloned policy (server-like) and register it as the best so far.
bc_score = evaluate(ExportNet(model.policy).to(device), n_episodes=N_EVAL_EPISODES)
update_best(model.policy, bc_score, "bc")


def plot_bc(run_dir=RUN_DIR):
    from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
    runs = sorted(glob.glob(os.path.join(run_dir, "bc_*")))
    if not runs:
        print("No BC run found yet."); return
    ea = EventAccumulator(runs[-1], size_guidance={"scalars": 0}); ea.Reload()
    avail = ea.Tags()["scalars"]

    def series(tag):
        ev = ea.Scalars(tag) if tag in avail else []
        return [e.step for e in ev], [e.value for e in ev]

    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    for tag, lbl in [("bc/train_loss", "train"), ("bc/val_loss", "val")]:
        x, y = series(tag)
        if x:
            ax[0].plot(x, y, label=lbl)
    ax[0].set(title="BC loss (cross-entropy)", xlabel="step"); ax[0].legend(); ax[0].grid(alpha=0.3)
    x, y = series("bc/val_acc")
    if x:
        ax[1].plot(x, y, color="green")
    ax[1].set(title="BC validation accuracy", xlabel="step", ylim=(0, 1)); ax[1].grid(alpha=0.3)
    fig.tight_layout(); plt.show()


plot_bc()

## Training loop

Fine-tune the behaviour-cloned policy with PPO on the 4-bot ROOM environment. The policy
is warm-started from the BC weights, and `target_kl` keeps each update close to the cloned
behaviour so PPO improves on it rather than overwriting it. Progress is logged to
TensorBoard (`runs/`); the best snapshot is evaluated and kept across runs.

In [ ]:
from stable_baselines3.common.callbacks import CheckpointCallback, EvalCallback, BaseCallback

ckpt_cb = CheckpointCallback(save_freq=CKPT_FREQ, save_path=CKPT_DIR, name_prefix="ppo_doom")

eval_env = make_doom_env(num_bots=NUM_BOTS, seed=4242, episode_timeout=EPISODE_TIMEOUT)
eval_cb = EvalCallback(
    eval_env,
    best_model_save_path=os.path.join(CKPT_DIR, "best"),
    log_path=os.path.join(CKPT_DIR, "best"),
    eval_freq=EVAL_FREQ,
    n_eval_episodes=N_EVAL_CB,
    deterministic=False,   # sample from the policy, like the server
    render=False,
)


class FragLogger(BaseCallback):
    def _on_rollout_end(self):
        buf = self.model.ep_info_buffer
        fr = [e["frags"] for e in buf if "frags" in e] if buf else []
        if fr:
            self.logger.record("rollout/frags_mean", float(np.mean(fr)))

    def _on_step(self):
        return True


callbacks = [ckpt_cb, eval_cb, FragLogger()]
print(f"callbacks ready | bots={NUM_BOTS} | eval_freq={EVAL_FREQ} | ckpt_freq={CKPT_FREQ}")

In [ ]:
# Warm-start PPO from the behaviour-cloned weights, then fine-tune.
if WARMSTART_FROM_BC and os.path.exists(BC_POLICY):
    load_policy(model.policy, BC_POLICY)
    print("warm-started PPO from", BC_POLICY)
else:
    print("training PPO from scratch (no BC warm-start found)")

total_timesteps = EPISODES * EPISODE_TIMEOUT
print(f"training PPO for {total_timesteps} steps")
model.learn(
    total_timesteps=total_timesteps,
    callback=callbacks,
    tb_log_name="ppo_doom",
    reset_num_timesteps=False,
)
model.save(os.path.join(CKPT_DIR, "ppo_final"))

# Evaluate the run's best snapshot (saved by EvalCallback) and keep it if it improves.
best_zip = os.path.join(CKPT_DIR, "best", "best_model.zip")
best_policy = PPO.load(best_zip, device=device).policy if os.path.exists(best_zip) else model.policy
ppo_score = evaluate(ExportNet(best_policy).to(device), n_episodes=N_EVAL_EPISODES)
update_best(best_policy, ppo_score, "ppo")

## Dump to ONNX

In [ ]:
# Export the best policy so far -> models/model.onnx (submit this file).
load_best(model.policy)
final_model = ExportNet(model.policy).to(device)
onnx_dump(env, final_model, PLAYER_CONFIG, filename="models/model.onnx")

### Evaluation and Visualization

In this final section, you can evaluate your trained agent, inspect its performance visually, and analyze reward components over time.

In [ ]:
# Training curves, read straight from the TensorBoard logs in runs/.
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator


def _read_scalars(logdir):
    ea = EventAccumulator(logdir, size_guidance={"scalars": 0}); ea.Reload()
    return {t: ([e.step for e in ea.Scalars(t)], [e.value for e in ea.Scalars(t)])
            for t in ea.Tags().get("scalars", [])}


def _smooth(y, k=9):
    y = np.asarray(y, dtype=float)
    if len(y) < k or k < 2:
        return y
    pad = np.r_[np.full(k - 1, y[0]), y]
    return np.convolve(pad, np.ones(k) / k, mode="valid")


def plot_training(run_dir=RUN_DIR):
    merged = {}
    for d in sorted(glob.glob(os.path.join(run_dir, "*"))):
        if not os.path.isdir(d):
            continue
        for tag, (xs, ys) in _read_scalars(d).items():
            mx, my = merged.setdefault(tag, ([], []))
            mx.extend(xs); my.extend(ys)
    panels = [
        ("rollout/ep_rew_mean", "Episode reward (train env)"),
        ("eval/mean_reward",    "Episode reward (4-bot eval)"),
        ("rollout/frags_mean",  "Frags / episode"),
        ("train/approx_kl",     "approx_kl (policy step size)"),
        ("train/value_loss",    "Value loss"),
        ("train/entropy_loss",  "Entropy loss"),
    ]
    avail = [(t, ti) for t, ti in panels if merged.get(t) and merged[t][1]]
    if not avail:
        print(f"No TensorBoard scalars under {run_dir}/ yet -- train first."); return
    rows = (len(avail) + 1) // 2
    fig, axes = plt.subplots(rows, 2, figsize=(13, 3.1 * rows))
    axes = np.atleast_1d(axes).flatten()
    for ax, (tag, title) in zip(axes, avail):
        xs, ys = merged[tag]
        order = np.argsort(xs)
        xs, ys = np.asarray(xs)[order], np.asarray(ys)[order]
        ax.plot(xs, ys, alpha=0.25, lw=1)
        ax.plot(xs, _smooth(ys), lw=2)
        ax.set_title(title, fontsize=10); ax.grid(alpha=0.3); ax.set_xlabel("step")
    for ax in axes[len(avail):]:
        ax.axis("off")
    fig.tight_layout(); plt.show()


plot_training()

In [ ]:
# Watch the final model play one 4-bot episode (inline video).
record_and_render(final_model, env, subsample=6)